# Smoke Test — Local Pipeline Execution

Runs `preprocess.py` → `train_mock.py` locally from JupyterLab (or any machine with S3 access).

**Intended environment:** SageMaker JupyterLab on `ml.c5.2xlarge`

This proves the pipeline scripts work end-to-end before running them as managed SageMaker jobs.

In [ ]:
import os
import shutil

# Local working directories (ephemeral — safe to delete)
BASE = "/tmp/ed_triage_smoke_test"

DIRS = {
    "input":      os.path.join(BASE, "input", "features"),
    "train":      os.path.join(BASE, "output", "train"),
    "validation": os.path.join(BASE, "output", "validation"),
    "test":       os.path.join(BASE, "output", "test"),
    "model":      os.path.join(BASE, "model"),
}

# Clean slate
if os.path.exists(BASE):
    shutil.rmtree(BASE)

for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)
    print(f"{name:12s} → {path}")


## Step 1: Download data from S3

In [ ]:
import boto3
import pandas as pd

S3_BUCKET = "ed-triage-capstone-group7"
S3_KEY    = "Data_Output/consolidated_dataset_features.csv"
LOCAL_CSV = os.path.join(DIRS["input"], "consolidated_dataset_features.csv")

s3 = boto3.client("s3", region_name="us-east-1")
s3.download_file(S3_BUCKET, S3_KEY, LOCAL_CSV)

df = pd.read_csv(LOCAL_CSV)
print(f"Downloaded: {df.shape[0]} rows × {df.shape[1]} cols")
print(f"File: {LOCAL_CSV}")


## Step 2: Run preprocessing

In [ ]:
import subprocess, sys

env = os.environ.copy()
env["PREPROCESS_INPUT_DIR"]    = DIRS["input"]
env["PREPROCESS_OUTPUT_TRAIN"] = DIRS["train"]
env["PREPROCESS_OUTPUT_VAL"]   = DIRS["validation"]
env["PREPROCESS_OUTPUT_TEST"]  = DIRS["test"]

result = subprocess.run(
    [sys.executable, "scripts/preprocess.py"],
    env=env,
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError(f"preprocess.py failed with exit code {result.returncode}")
print("✓ Preprocessing completed successfully")


In [ ]:
# Verify splits
train_df = pd.read_csv(os.path.join(DIRS["train"], "train.csv"))
val_df   = pd.read_csv(os.path.join(DIRS["validation"], "val.csv"))
test_df  = pd.read_csv(os.path.join(DIRS["test"], "test.csv"))

print(f"Train: {train_df.shape}  Val: {val_df.shape}  Test: {test_df.shape}")
print(f"Total rows: {len(train_df) + len(val_df) + len(test_df)} (expected {len(df)})")
print(f"\nTrain class distribution:")
print(train_df["triage_3class"].value_counts().sort_index().to_string())
print(f"\nColumns: {list(train_df.columns)}")

assert len(train_df) + len(val_df) + len(test_df) == len(df), "Row count mismatch!"
assert "triage_text" in train_df.columns, "triage_text column missing!"
assert "triage_3class" in train_df.columns, "triage_3class column missing!"
print("\n✓ All split checks passed")


## Step 3: Run mock training

In [ ]:
env2 = os.environ.copy()
env2["SM_CHANNEL_TRAIN"]      = DIRS["train"]
env2["SM_CHANNEL_VALIDATION"] = DIRS["validation"]
env2["SM_MODEL_DIR"]          = DIRS["model"]
env2["SM_HP_EPOCHS"]          = "1"

result = subprocess.run(
    [sys.executable, "scripts/train_mock.py"],
    env=env2,
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError(f"train_mock.py failed with exit code {result.returncode}")
print("✓ Mock training completed successfully")


In [ ]:
# Verify training artifacts
model_path = os.path.join(DIRS["model"], "model.pt")
stats_path = os.path.join(DIRS["model"], "structured_stats.json")

artifacts = {
    "model.pt": model_path,
    "structured_stats.json": stats_path,
}

all_ok = True
print("Training artifacts:")
for name, path in artifacts.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = f"{size:,} bytes" if exists else "MISSING"
    print(f"  {name:30s} {status}")
    if not exists:
        all_ok = False

# Also show split files
print("\nSplit files:")
for split_name, split_dir in [("train", DIRS["train"]), ("validation", DIRS["validation"]), ("test", DIRS["test"])]:
    for f in os.listdir(split_dir):
        size = os.path.getsize(os.path.join(split_dir, f))
        print(f"  {split_name}/{f:20s} {size:,} bytes")

if all_ok:
    print("\n" + "=" * 50)
    print("  SMOKE TEST PASSED — all artifacts produced")
    print("=" * 50)
else:
    raise RuntimeError("Smoke test FAILED — missing artifacts")
